# Exercise: Feed Clean Documents into Your RAG Pipeline

In the walkthrough you built the RAG pipeline by hand on text pulled straight from a PDF reader. Real documents are messier than that, and the quality of the text you feed in directly shapes the quality of what you retrieve.

In this exercise you will reuse the previous pipeline but we convert the document to Markdown with
[MarkItDown](https://github.com/microsoft/markitdown), clean it, and then **measure** how much that cleaning changes what the pipeline retrieves and answers.

## Learning Objectives

By completing this exercise, you will be able to:

- Convert a PDF to Markdown with MarkItDown and inspect the raw conversion.
- Write cleaning rules that remove common extraction artifacts.
- Measure the effect of cleaning on retrieval with similarity scores and chunk overlap.

## Setup

Load your credentials. The Groq client reads `GROQ_API_KEY` from your `.env` file.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

PDF = "documents/react_paper.pdf"
QUERY = "What is the ReAct framework and how does it combine reasoning and acting?"

## Provided: the RAG pipeline from the walkthrough

Run this cell as is. It gives you two things:

- `run_rag(text, query)`, which chunks a body of text, embeds it, retrieves the most
  relevant chunks, and generates an answer. It returns the `answer`, the retrieved
  `chunks`, and their cosine similarity `scores`.
- `retrieval_overlap(chunks_a, chunks_b)`, the Jaccard overlap between two sets of
  retrieved chunks (0 means completely different, 1 means identical).

You do not need to change anything here.

In [ ]:
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


def chunk_text(text, size=800, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


def retrieve(query, chunks, chunk_vectors, k=3):
    query_vector = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ query_vector
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]


def generate_answer(query, context):
    prompt = (
        "Answer the question using only the context below.\n\n"
        "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {query}\nAnswer:"
    )
    response = Groq().chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return response.choices[0].message.content


def run_rag(text, query, k=3):
    """Chunk, embed, retrieve, and generate. Returns answer, chunks, and scores."""
    chunks = chunk_text(text)
    chunk_vectors = embedder.encode(chunks, normalize_embeddings=True)
    hits = retrieve(query, chunks, chunk_vectors, k=k)
    context = [chunk for chunk, _ in hits]
    return {
        "answer": generate_answer(query, context),
        "chunks": context,
        "scores": [score for _, score in hits],
    }


def retrieval_overlap(chunks_a, chunks_b):
    """Jaccard overlap between two lists of retrieved chunks (0 to 1)."""
    a, b = set(chunks_a), set(chunks_b)
    return len(a & b) / len(a | b) if (a or b) else 0.0

## Exercise 1: Convert the document to Markdown

Use MarkItDown to convert the PDF to a Markdown string.

Hints:

- The API is `MarkItDown().convert(path).text_content`.
- After converting, print the first 1000 characters and look closely. Real PDF
  extractions often have missing spaces, repeated page headers, and stray line breaks.
  Note the problems you see: you will fix them next.

In [ ]:
# TODO: import MarkItDown from markitdown


def to_markdown(path):
    """Convert a document to a Markdown string with MarkItDown."""
    # TODO: instantiate MarkItDown, convert the file, and return the text content

In [ ]:
# TODO: convert PDF, store the result as `raw_text`, and print the first 1000 characters

## Exercise 2: Clean the Markdown

Write a cleaning function that fixes the problems you spotted. Decide which rules
matter for this document.

Ideas to consider:

- Collapse runs of blank lines and strip trailing whitespace.
- Drop lines that are just a page number or the repeated conference header.
- Remove very short lines that are page furniture rather than content.
- Rejoin words split across a line break by a hyphen.

Store the result as `clean_text` and print the character count before and after so you
can see how much you removed.

In [ ]:
# TODO: import re


def clean_markdown(text):
    """Return a cleaned version of the Markdown text."""
    # TODO: apply your cleaning rules and return the result

In [ ]:
# TODO: clean `raw_text` into `clean_text` and compare the length before and after

## Exercise 3: Measure the effect of cleaning

This is the payoff. Run the provided pipeline twice on the same question, once with
`raw_text` and once with `clean_text`, then compare them with two metrics.

**Metric 1: mean retrieval score.** Average the `scores` of the retrieved chunks for
each run. A higher mean suggests the retrieved chunks match the query more closely.

**Metric 2: retrieval overlap.** Use `retrieval_overlap(...)` on the two `chunks`
lists. A low overlap means cleaning changed *which* chunks were retrieved.

> Read the scores as a signal, not a verdict. The raw and cleaned runs have different
> chunk sets, so the numbers are not a perfect like-for-like comparison. Always read the
> retrieved chunks and the answer alongside the metrics.

Tasks:

- Call `run_rag` on `raw_text` and on `clean_text`.
- Print the mean retrieval score for each run and the overlap between them.
- Print both answers, then write a sentence on what changed and why.

In [ ]:
# TODO: run_rag on raw_text and clean_text
# TODO: print mean(scores) for each run, and retrieval_overlap(raw_chunks, clean_chunks)
# TODO: print both answers and note what changed

## Exercise 4 (stretch): Score the answers with an LLM judge

Similarity scores tell you about retrieval, not about the final answer. A common way to
score answer quality without a reference answer is to use the model as a judge.

Write a function that asks the model to rate, from 1 to 5, how well an answer is
supported by the context it was given (its faithfulness). Run it for both the raw and
the cleaned answer and compare.

Hint: build a prompt that includes the question, the context, and the answer, and ask
the model to reply with a single number. Reuse the `Groq` client from the provided cell.

In [ ]:
def judge_faithfulness(question, context, answer):
    """Ask the model to rate 1 to 5 how well the answer is supported by the context."""
    # TODO: build a judging prompt, call Groq, and return the score

In [ ]:
# TODO: judge the raw answer and the cleaned answer, then compare the two scores

## Conclusion and Reflection

- Which cleaning rule made the biggest difference to the mean retrieval score or the
  overlap, and why?
- Did cleaning change the answer, or only the chunks behind it? What does that tell you
  about how sensitive this question is to noise?
- The mean score is an imperfect metric. What would you need in order to measure answer
  quality properly, and which tools (for example RAGAS) formalise this?